In [ ]:
import os, numpy as np, pandas as pd
import statsmodels.api as sm
from IPython.display import display

pd.set_option("display.float_format", lambda x: f"{x:,.1f}")

# ── BC 소비데이터 ──
# 코드값은 문자열로 고정해야 조인 시 타입이 맞는다
bc = pd.read_csv("data/ABP_CONTEST_DATA.csv", encoding="utf-8",
                 dtype={"STRD_YYMM": str, "GENDER_CD": str,
                        "AGE_CD": str, "TP_BUZ_NO": str})

# ── 인구 (population.ipynb 결과물) ──
pop = pd.read_csv("data/인구_시군구_성별_연령_202601_202606.csv", encoding="utf-8-sig",
                  dtype={"STRD_YYMM": str, "GENDER_CD": str,
                         "AGE_CD": str, "행정구역코드": str})

# ── 업소수 (수집 캐시, API 호출 없음) ──
업소_8업종 = pd.read_csv("data/cache/업소수_전국시군구.csv", encoding="utf-8-sig",
                     dtype={"TP_BUZ_NO": str})
대규모 = pd.read_csv("data/cache/대규모점포_원본.csv", encoding="utf-8-sig", dtype=str)

print("BC      :", bc.shape)
print("인구    :", pop.shape)
print("업소수  :", 업소_8업종.shape, f"({업소_8업종['행정구역명'].nunique()}개 지역)")
print("대규모점포:", 대규모.shape)


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"      # 윈도우 한글 폰트
plt.rcParams["axes.unicode_minus"] = False

파랑, 주황 = "#2a78d6", "#eb6834"
잉크, 보조, 흐림 = "#0b0b0b", "#52514e", "#898781"
격자, 바탕 = "#e1e0d9", "#fcfcfb"

# 업종별 전국 BC 소비액 — 우리가 다루는 시장이 어떤 규모인지 감을 잡는다
업종규모 = (bc.groupby("TP_BUZ_NM")["amt"].sum() / 1e12).sort_values()

fig, ax = plt.subplots(figsize=(8, 4.5), facecolor=바탕)
ax.set_facecolor(바탕)
ax.barh(업종규모.index, 업종규모.values, color=파랑, height=0.65)

# 막대 끝에 값을 직접 적는다 (축 눈금을 읽게 하지 않는다)
for i, v in enumerate(업종규모.values):
    ax.text(v + 0.05, i, f"{v:.2f}조", va="center", color=보조, fontsize=9.5)

ax.set_title("BC 소비데이터 업종별 규모 (2026년 1~6월 전국)",
             color=잉크, fontsize=13, fontweight="bold", loc="left", pad=12)
ax.set_xlim(0, 업종규모.max() * 1.18)
ax.tick_params(colors=흐림, labelsize=10, length=0)
ax.grid(axis="x", color=격자, lw=1)
ax.set_axisbelow(True)
for s in ["top", "right", "left", "bottom"]:
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# ── BC: 시도 + 시군구를 한 칸으로 합치기 ──
bc["행정구역명"] = bc["SIDO_NM"].str.strip() + " " + bc["CCG_NM"].str.strip()

# 세종시는 BC에서 "세종특별자치시 세종특별자치시"로 중복되므로 인구 쪽 표기로 맞춘다
bc["행정구역명"] = bc["행정구역명"].replace("세종특별자치시 세종특별자치시", "세종특별자치시")

# 공백이 두 칸 이상 들어간 경우 대비
bc["행정구역명"] = bc["행정구역명"].str.replace(r"\s+", " ", regex=True).str.strip()


# ── 인천 개편 보정 ──
# 상가정보는 개편 후(제물포·영종·서해·검단), BC·인구는 개편 전(중구·동구·서구) 체계다.
#   중구 + 동구  ↔  제물포구 + 영종구     ← 1:1이 아니라 합집합끼리만 일치
#   서구         ↔  서해구 + 검단구       ← 서구 이름은 그대로 쓸 수 있다
# 따라서 중구·동구를 하나로 묶어 "인천광역시 중구+동구"라는 단위를 만든다.
인천병합 = {"인천광역시 중구": "인천광역시 중구+동구",
         "인천광역시 동구": "인천광역시 중구+동구"}

def 지역정리(df):
    d = df.copy()
    d["행정구역명"] = d["행정구역명"].replace(인천병합)
    return d

bc = 지역정리(bc)
pop = 지역정리(pop)

# ── 매칭 확인 ──
print("BC 지역 수  :", bc["행정구역명"].nunique())
print("인구 지역 수:", pop["행정구역명"].nunique())
print("업소수 지역 :", 업소_8업종["행정구역명"].nunique())
print()
print("BC에만 있음  :", sorted(set(bc["행정구역명"]) - set(pop["행정구역명"])))
print("인구에만 있음:", sorted(set(pop["행정구역명"]) - set(bc["행정구역명"])))


In [ ]:
# 255개 지역이 227개로 줄어드는 과정 — 무엇이 왜 빠지는지 눈으로 확인
단계 = ["전체 시군구", "광주 제외", "전남 제외", "인천 중구+동구 병합"]
값   = [255, 255-5, 255-5-22, 255-5-22-1]
설명 = ["", "-5\n상가정보 없음", "-22\n상가정보 없음", "-1\n행정구역 개편"]

fig, ax = plt.subplots(figsize=(8.5, 4.2), facecolor=바탕)
ax.set_facecolor(바탕)

색 = [파랑, 흐림, 흐림, 주황]
bars = ax.bar(단계, 값, color=색, width=0.55)

for b, v, s in zip(bars, 값, 설명):
    ax.text(b.get_x() + b.get_width()/2, v + 5, f"{v}개",
            ha="center", color=잉크, fontsize=11, fontweight="bold")
    if s:
        ax.text(b.get_x() + b.get_width()/2, v/2, s,
                ha="center", va="center", color="white", fontsize=9.5, linespacing=1.5)

ax.set_title("분석 대상 지역이 227개인 이유",
             color=잉크, fontsize=13, fontweight="bold", loc="left", pad=12)
ax.set_ylabel("시군구 수", color=보조, fontsize=10)
ax.set_ylim(0, 290)
ax.tick_params(colors=흐림, labelsize=10, length=0)
ax.grid(axis="y", color=격자, lw=1)
ax.set_axisbelow(True)
for s_ in ["top", "right", "left"]:
    ax.spines[s_].set_visible(False)
ax.spines["bottom"].set_color(격자)
plt.tight_layout()
plt.show()

print("제외되는 인구:", "316만명 (전국의 6.2%) — 결과물에 한계로 명시할 것")


In [ ]:
# 제외되는 인구 규모 — 하드코딩하지 않고 매번 계산한다
#   데이터가 갱신되면 이 숫자도 따라 바뀌어야 하므로
최근월 = pop["STRD_YYMM"].max()
그달 = pop[pop["STRD_YYMM"] == 최근월]
제외 = 그달[그달["행정구역명"].str.startswith(("광주광역시", "전라남도"))]

print(f"제외 지역: {제외['행정구역명'].nunique()}개")
print(f"제외 인구: {제외['인구'].sum():,}명 "
      f"(전국 {그달['인구'].sum():,}명의 {제외['인구'].sum()/그달['인구'].sum()*100:.1f}%) "
      f"— {최근월} 기준")
print("→ 결과물에 한계로 명시할 것")
    

상가정보 API가 행정구역 통합 작업 중이라 그 지역 데이터를 아예 안 주기 때문

In [ ]:
# ══ 3-1. 대형마트 지역 매칭 ══
# 대규모점포 데이터는 2026년 8월 기준이라 주소가 개편된 행정구역을 쓴다.
# BC·인구(2026.6)와 맞추려면 옛 체계로 되돌려야 한다.

신구 = {"인천광역시 제물포구": "인천광역시 중구+동구",
      "인천광역시 영종구":   "인천광역시 중구+동구",
      "인천광역시 서해구":   "인천광역시 서구",
      "인천광역시 검단구":   "인천광역시 서구",
      "인천광역시 중구":     "인천광역시 중구+동구",
      "인천광역시 동구":     "인천광역시 중구+동구"}

# 긴 이름부터 찾아야 '수원시'가 '수원시 팔달구'를 가로채지 않는다
매칭목록 = sorted(set(pop["행정구역명"]) | set(신구), key=len, reverse=True)

주소 = (대규모["LOTNO_ADDR"].fillna("") + " " + 대규모["ROAD_NM_ADDR"].fillna(""))
주소 = (주소
    .str.replace(r"전남광주통합특별시(\s+[가-힣]+구)", r"광주광역시\1", regex=True)  # 통합 전으로 복원
    .str.replace("전남광주통합특별시", "전라남도", regex=False)
    .str.replace("인천광역시 남구", "인천광역시 미추홀구", regex=False)   # 2018년 개칭된 옛 이름
    .str.replace(r"([가-힣]{2,3}시)([가-힣]+구)", r"\1 \2", regex=True))  # '고양시일산동구' 공백 보정

대규모["행정구역명"] = 주소.apply(
    lambda a: next((g for g in 매칭목록 if g in a), None)).replace(신구)

print("1차 주소 매칭 실패:", 대규모["행정구역명"].isna().sum(), "건")


In [ ]:
# ══ 3-2. 주소가 깨진 건을 개방자치단체코드로 메우기 ══
# 일부 점포는 주소가 '133번지 1호'처럼 불완전하다.
# 같은 개방자치단체코드를 가진 다른 점포의 지역을 빌려온다.
#
# 주의: 개방코드는 '시' 단위라서 분구 도시(창원 5개 구 등)에서는 구를 특정할 수 없다.
#       그래서 코드가 단 하나의 지역만 가리킬 때만 쓴다.
성공 = 대규모.dropna(subset=["행정구역명"])
지역수 = 성공.groupby("OPN_ATMY_GRP_CD")["행정구역명"].nunique()
단일코드 = 성공.groupby("OPN_ATMY_GRP_CD")["행정구역명"].first()[지역수 == 1]

결측 = 대규모["행정구역명"].isna()
대규모.loc[결측, "행정구역명"] = 대규모.loc[결측, "OPN_ATMY_GRP_CD"].map(단일코드)

# 화성 봉담점 2건만 남는데, 같은 개방코드의 다른 점포 주소가
# '경기도 화성시 효행구 봉담읍 상리'여서 봉담읍 = 효행구임이 확인된다
대규모.loc[대규모["행정구역명"].isna() &
        대규모["ROAD_NM_ADDR"].fillna("").str.contains("효행로"), "행정구역명"] = "경기도 화성시 효행구"

# 영업 중인 대형마트만 (BC의 '대형할인점'에 대응)
마트 = 대규모[(대규모["SALS_STTS_NM"] == "영업/정상") &
           (대규모["BZSTAT_SE_NM"] == "대형마트")].copy()

print("영업 중 대형마트:", len(마트), "개 / 매칭 실패:", 마트["행정구역명"].isna().sum())
print("대형마트 보유 지역:", 마트["행정구역명"].nunique(), "개")


In [ ]:
# ══ 3-3. 세 데이터를 한 테이블로 ══
성인 = ["2", "3", "4", "5", "6"]          # 0~19세 제외 (카드를 거의 안 쓴다)
제외업종 = ["8002", "8003"]                # 갈비전문점·한정식 (표본이 너무 적다)

# 분자: BC 소비 — 내국인 + 외국인, 법인만 제외
#   업소수가 내·외국인 구분 없는 전체 점포이므로 분자도 범위를 맞춘다 (기획서 3단계 지침)
소비 = (bc[bc["GENDER_CD"].isin(["1", "2", "3"]) & ~bc["TP_BUZ_NO"].isin(제외업종)]
      .groupby(["행정구역명", "TP_BUZ_NO", "TP_BUZ_NM"], as_index=False)["amt"].sum())

# 업소수: 상가정보 8개 업종 + 대규모점포에서 온 대형할인점
마트수 = 마트.groupby("행정구역명").size().rename("업소수").reset_index()
마트수["TP_BUZ_NO"] = "4004"
업소 = pd.concat([업소_8업종, 마트수], ignore_index=True)

# 인구: 20대 이상 (6개월 평균)
인구 = (pop[pop["AGE_CD"].isin(성인)]
      .groupby(["행정구역명", "STRD_YYMM"])["인구"].sum()
      .groupby("행정구역명").mean().rename("성인인구").reset_index())

# 합치기 — 세 데이터에 모두 있는 조합만 남는다(inner)
분석 = (소비.merge(업소, on=["행정구역명", "TP_BUZ_NO"], how="inner")
      .merge(인구, on="행정구역명", how="inner"))

# 로그를 씌울 수 없는 0은 제외
분석 = 분석[(분석["업소수"] > 0) & (분석["amt"] > 0)].copy()

print("분석 테이블:", 분석.shape)
print("지역:", 분석["행정구역명"].nunique(), "개 / 업종:", 분석["TP_BUZ_NO"].nunique(), "개")
display(분석.head())


In [ ]:
# 업종별로 몇 개 지역이 들어왔는지 — 대형할인점만 유독 적은 이유를 눈으로 확인
커버 = 분석.groupby("TP_BUZ_NM").size().sort_values()

fig, ax = plt.subplots(figsize=(8, 4.2), facecolor=바탕)
ax.set_facecolor(바탕)
색 = [주황 if v < 200 else 파랑 for v in 커버.values]
ax.barh(커버.index, 커버.values, color=색, height=0.65)
for i, v in enumerate(커버.values):
    ax.text(v + 2, i, f"{v}개 지역", va="center", color=보조, fontsize=9.5)

ax.axvline(227, color=흐림, ls="--", lw=1)
ax.text(227, -0.9, "전체 227", color=흐림, fontsize=9, ha="center")

ax.set_title("업종별 분석 대상 지역 수", color=잉크, fontsize=13,
             fontweight="bold", loc="left", pad=12)
ax.set_xlim(0, 260)
ax.tick_params(colors=흐림, labelsize=10, length=0)
ax.grid(axis="x", color=격자, lw=1)
ax.set_axisbelow(True)
for s_ in ["top", "right", "left", "bottom"]:
    ax.spines[s_].set_visible(False)
plt.tight_layout()
plt.show()


대형할인점만 153개 지역으로 뚝 떨어집니다. 나머지 74개 지역엔 대형마트가 아예 없어서입니다. 이게 나중에 "대형할인점 결과는 신뢰도가 낮다"고 말해야 하는 이유 중 하나입니다.

Step 4. 회귀 모델


Step 4-1. 회귀 돌리기



In [ ]:
# 로그를 씌우는 이유
#  - 금액·점포수는 큰 도시와 작은 군의 차이가 수백 배라 그대로 쓰면 큰 도시가 모델을 지배한다
#  - 로그를 씌우면 "몇 배 차이"로 바뀌어 작은 지역도 공평하게 반영된다
#  - 계수도 "점포 1% 늘면 소비 몇 % 느는가"로 읽을 수 있다
분석["log_소비"]   = np.log(분석["amt"])
분석["log_업소수"] = np.log(분석["업소수"])
분석["log_인구"]   = np.log(분석["성인인구"])

설명변수 = ["log_업소수", "log_인구"]

요약, 잔차모음 = [], []
for 업종코드, d in 분석.groupby("TP_BUZ_NO"):
    X = sm.add_constant(d[설명변수])

    # cov_type="HC3" : 로버스트 표준오차.
    #   작은 지역일수록 예측이 더 많이 빗나가는 성질(이분산)이 있어
    #   일반 표준오차를 쓰면 p값을 과신하게 된다.
    모델 = sm.OLS(d["log_소비"], X).fit(cov_type="HC3")

    # 예측값과 실제값의 비율 = 침투지수
    #   100이면 예측대로, 50이면 예측의 절반만 결제된 것
    d = d.assign(예측=np.exp(모델.fittedvalues))
    d["침투지수"] = d["amt"] / d["예측"] * 100
    잔차모음.append(d)

    요약.append({"TP_BUZ_NO": 업종코드, "TP_BUZ_NM": d["TP_BUZ_NM"].iat[0],
               "표본수": len(d), "R2": 모델.rsquared,
               "업소수계수": 모델.params["log_업소수"], "p_업소수": 모델.pvalues["log_업소수"],
               "인구계수": 모델.params["log_인구"], "p_인구": 모델.pvalues["log_인구"]})

회귀결과 = pd.concat(잔차모음, ignore_index=True)
모델요약 = pd.DataFrame(요약).sort_values("R2", ascending=False)

with pd.option_context("display.float_format", "{:.3f}".format):
    display(모델요약)


Step 4-2. 모델 신뢰도 한눈에 보기



In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2), facecolor=바탕)
ax.set_facecolor(바탕)

s = 모델요약.sort_values("R2")
색 = [주황 if v < 0.7 else 파랑 for v in s["R2"]]
ax.barh(s["TP_BUZ_NM"], s["R2"], color=색, height=0.65)

for i, (v, n) in enumerate(zip(s["R2"], s["표본수"])):
    ax.text(v + 0.012, i, f"{v:.2f}  (n={n})", va="center", color=보조, fontsize=9.5)

ax.axvline(0.7, color=흐림, ls="--", lw=1)
ax.set_title("업종별 모델 설명력 (R²)", color=잉크, fontsize=13,
             fontweight="bold", loc="left", pad=12)
ax.set_xlim(0, 1.15)
ax.tick_params(colors=흐림, labelsize=10, length=0)
ax.grid(axis="x", color=격자, lw=1)
ax.set_axisbelow(True)
for s_ in ["top", "right", "left", "bottom"]:
    ax.spines[s_].set_visible(False)
plt.tight_layout()
plt.show()


Step 5. 시장규모를 BC와 무관하게 재기

추정 시장규모 = 그 지역 점포 수 × 전국 점포당 평균 BC 소비


In [ ]:
# 업종별 전국 점포당 평균 소비 (6개월 기준)
#  '그 지역'이 아니라 '전국' 평균을 쓰는 게 핵심 — 지역 실적이 섞이면 안 된다
전국업소당 = (회귀결과.groupby("TP_BUZ_NO")
          .apply(lambda d: d["amt"].sum() / d["업소수"].sum(), include_groups=False)
          .rename("전국업소당소비"))

회귀결과 = 회귀결과.merge(전국업소당, on="TP_BUZ_NO")

# 추정 시장규모 = 점포 수 × 전국 평균
#  "이 지역에서 BC가 다 가져간다면 이만큼" 이라는 상한선 개념
회귀결과["추정시장규모"] = 회귀결과["업소수"] * 회귀결과["전국업소당소비"]

# 업종 내 시장규모 상위 50%만 후보로
#  업종마다 규모가 10배 이상 차이나므로 전체가 아니라 업종 안에서 비교한다
회귀결과["시장중앙값"] = 회귀결과.groupby("TP_BUZ_NO")["추정시장규모"].transform("median")
후보 = 회귀결과[회귀결과["추정시장규모"] >= 회귀결과["시장중앙값"]].copy()

# 놓치고 있는 금액 = 시장규모 - 실제 BC 소비
회귀결과["미확보금액"] = 회귀결과["추정시장규모"] - 회귀결과["amt"]
후보["미확보금액"] = 후보["추정시장규모"] - 후보["amt"]

print(f"전체 {len(회귀결과)}개 조합 → 시장규모 상위 50%: {len(후보)}개")


In [ ]:
display(후보.nsmallest(15, "침투지수")
        .assign(실제_억=lambda d: d["amt"] / 1e8,
                시장_억=lambda d: d["추정시장규모"] / 1e8,
                미확보_억=lambda d: d["미확보금액"] / 1e8)
        [["행정구역명", "TP_BUZ_NM", "업소수", "실제_억", "시장_억", "미확보_억", "침투지수"]])


Step 5 코드 (대형할인점 제외)

In [ ]:
# 대형할인점 제외 — 상권이 행정구역을 넘어서 점포 수·면적 모두 소비를 설명하지 못한다
#   업소수 계수 0.28, 면적으로 바꿔도 R² 0.468 → 0.467로 제자리
#   결국 인구 대비 소비를 재게 되는데, 그건 우리가 처음에 버린 지표와 같은 약점을 갖는다
회귀결과 = 회귀결과[회귀결과["TP_BUZ_NO"] != "4004"].copy()

# 시장규모 상위 50% (업종 내 기준)
회귀결과["시장중앙값"] = 회귀결과.groupby("TP_BUZ_NO")["추정시장규모"].transform("median")
후보 = 회귀결과[회귀결과["추정시장규모"] >= 회귀결과["시장중앙값"]].copy()
후보["미확보금액"] = 후보["추정시장규모"] - 후보["amt"]

print(f"8개 업종 {len(회귀결과)}개 조합 → 시장규모 상위 50%: {len(후보)}개")


In [ ]:
파랑, 주황 = "#2a78d6", "#eb6834"
초록 = "#1baf7a"                                    # 세 번째 계열용
잉크, 보조, 흐림 = "#0b0b0b", "#52514e", "#898781"
격자, 축선, 바탕 = "#e1e0d9", "#c3c2b7", "#fcfcfb"


In [ ]:
# 두 관점이 얼마나 다른지 그림으로
fig, ax = plt.subplots(figsize=(9, 5.6), facecolor=바탕)
ax.set_facecolor(바탕)

유효 = 후보[후보["침투지수"] < 200]
ax.scatter(유효["미확보금액"] / 1e8, 유효["침투지수"],
           s=26, color=파랑, alpha=0.4, edgecolor="none", zorder=3)

# 두 기준의 상위 5개를 각각 강조
비율최악 = 후보.nsmallest(5, "침투지수")
금액최대 = 후보[후보["침투지수"] < 100].nlargest(5, "미확보금액")

for d_, c_, lab in [(비율최악, 주황, "비율 최악"), (금액최대, "#1baf7a", "금액 최대")]:
    ax.scatter(d_["미확보금액"] / 1e8, d_["침투지수"], s=80,
               facecolor="none", edgecolor=c_, lw=2, zorder=5, label=lab)
    for _, r in d_.iterrows():
        ax.annotate(f"{r['행정구역명'].split()[-1]} {r['TP_BUZ_NM']}",
                    (r["미확보금액"] / 1e8, r["침투지수"]),
                    textcoords="offset points", xytext=(8, 3),
                    fontsize=9, color=c_, fontweight="bold")

ax.axhline(100, color=흐림, ls="--", lw=1)
ax.set_title("두 관점은 서로 다른 곳을 가리킨다",
             color=잉크, fontsize=13.5, fontweight="bold", loc="left", pad=14)
ax.set_xlabel("미확보 금액 (억원) — 놓치고 있는 돈의 크기", color=보조, fontsize=10)
ax.set_ylabel("침투지수 — 낮을수록 상대적으로 약함", color=보조, fontsize=10)
ax.tick_params(colors=흐림, labelsize=9.5, length=0)
ax.grid(color=격자, lw=1)
ax.set_axisbelow(True)
for s_ in ["top", "right"]:
    ax.spines[s_].set_visible(False)
for s_ in ["left", "bottom"]:
    ax.spines[s_].set_color(축선)
ax.legend(frameon=False, loc="upper right", fontsize=10, labelcolor=보조)
plt.tight_layout()
plt.show()


Step 6 코드

In [ ]:
# 침투지수 100 미만 = 예측보다 적게 결제된 조합만
후보 = 후보[후보["침투지수"] < 100].copy()
print("후보풀:", len(후보), "개")

# ── 파레토 최적 ──
# 미확보금액이 더 크면서 침투지수도 더 낮은 조합이 없는 것 = 누구에게도 지배당하지 않음
#   미확보금액 큰 순으로 훑으며, 지금까지 본 것보다 침투지수가 낮으면 프론티어에 추가
s = 후보.sort_values("미확보금액", ascending=False).reset_index(drop=True)
최저 = np.inf
프론티어 = []
for i, r in s.iterrows():
    if r["침투지수"] < 최저:
        프론티어.append(i)
        최저 = r["침투지수"]

파레토 = s.loc[프론티어].sort_values("침투지수")
print("파레토 최적:", len(파레토), "개")

display(파레토.assign(실제_억=lambda d: d["amt"] / 1e8,
                    미확보_억=lambda d: d["미확보금액"] / 1e8)
        [["행정구역명", "TP_BUZ_NM", "업소수", "실제_억", "미확보_억", "침투지수"]])


In [ ]:
# ── 민감도 분석 ──
# 가중치를 0~100%까지 흔들며 상위 10에 몇 번 드는지 센다
#   특정 가중치에서만 반짝하는 후보와, 어떤 기준으로도 살아남는 후보를 구분한다
def 정규화(x):
    return (x - x.min()) / (x.max() - x.min())

후보["점수_시장"] = 정규화(후보["미확보금액"])      # 놓치는 돈이 클수록 높음
후보["점수_격차"] = 정규화(100 - 후보["침투지수"])  # 침투지수가 낮을수록 높음

진입횟수 = {}
for w in np.arange(0, 1.01, 0.05):
    상위 = 후보.assign(점수=w * 후보["점수_시장"] + (1 - w) * 후보["점수_격차"]).nlargest(10, "점수")
    for _, r in 상위.iterrows():
        키 = (r["행정구역명"], r["TP_BUZ_NM"])
        진입횟수[키] = 진입횟수.get(키, 0) + 1

안정성 = (pd.DataFrame([{"행정구역명": k[0], "TP_BUZ_NM": k[1], "진입횟수": v}
                     for k, v in 진입횟수.items()])
        .sort_values("진입횟수", ascending=False))

display(안정성.head(15))


In [ ]:
# ── 파레토 프론티어 그림 ──
fig, ax = plt.subplots(figsize=(9, 5.6), facecolor=바탕)
ax.set_facecolor(바탕)

ax.scatter(후보["미확보금액"] / 1e8, 후보["침투지수"],
           s=24, color=파랑, alpha=0.3, edgecolor="none", zorder=3, label="후보풀")

ax.plot(파레토["미확보금액"] / 1e8, 파레토["침투지수"],
        color=주황, lw=2, marker="o", ms=9, mfc=바탕, mew=2.2,
        zorder=5, label="파레토 최적")

for _, r in 파레토.iterrows():
    ax.annotate(f"{r['행정구역명'].split()[-1]}\n{r['TP_BUZ_NM']}",
                (r["미확보금액"] / 1e8, r["침투지수"]),
                textcoords="offset points", xytext=(10, 6),
                fontsize=9.5, color=주황, fontweight="bold", linespacing=1.3)

ax.set_title("가중치를 정하지 않아도 남는 후보들",
             color=잉크, fontsize=13.5, fontweight="bold", loc="left", pad=14)
ax.set_xlabel("미확보 금액 (억원) →  클수록 좋음", color=보조, fontsize=10)
ax.set_ylabel("침투지수 ↓  낮을수록 좋음", color=보조, fontsize=10)
ax.tick_params(colors=흐림, labelsize=9.5, length=0)
ax.grid(color=격자, lw=1)
ax.set_axisbelow(True)
for s_ in ["top", "right"]:
    ax.spines[s_].set_visible(False)
for s_ in ["left", "bottom"]:
    ax.spines[s_].set_color(축선)
ax.legend(frameon=False, loc="upper left", fontsize=10, labelcolor=보조)
plt.tight_layout()
plt.show()


최종 후보 확정



In [ ]:
# ── 파레토 최적 ──
# 미확보금액이 더 크면서 침투지수도 더 낮은 조합이 없는 것 = 누구에게도 지배당하지 않음
s = 후보.sort_values("미확보금액", ascending=False).reset_index(drop=True)
최저 = np.inf
프론티어 = []
for i, r in s.iterrows():
    if r["침투지수"] < 최저:
        프론티어.append(i)
        최저 = r["침투지수"]
파레토키 = set(zip(s.loc[프론티어, "행정구역명"], s.loc[프론티어, "TP_BUZ_NM"]))

# ── 민감도: 가중치를 0~100%까지 21가지로 흔들며 상위 10 진입 횟수 ──
def 정규화(x):
    return (x - x.min()) / (x.max() - x.min())

후보 = 후보.copy()
후보["점수_시장"] = 정규화(후보["미확보금액"])       # 놓치는 돈이 클수록 높음
후보["점수_격차"] = 정규화(100 - 후보["침투지수"])   # 침투지수가 낮을수록 높음

진입 = {}
for w in np.arange(0, 1.01, 0.05):
    상위 = 후보.assign(점수=w * 후보["점수_시장"] + (1 - w) * 후보["점수_격차"]).nlargest(10, "점수")
    for _, r in 상위.iterrows():
        키 = (r["행정구역명"], r["TP_BUZ_NM"])
        진입[키] = 진입.get(키, 0) + 1

# ── 두 기준을 합쳐 최종 후보 ──
#   파레토 최적이거나, 21가지 가중치 중 13번 이상 상위 10위에 든 조합
후보["키"] = list(zip(후보["행정구역명"], 후보["TP_BUZ_NM"]))
후보["파레토"] = 후보["키"].isin(파레토키)
후보["진입횟수"] = 후보["키"].map(진입).fillna(0).astype(int)

최종후보 = 후보[후보["파레토"] | (후보["진입횟수"] >= 13)].sort_values("침투지수").copy()

print(f"후보풀 {len(후보)}개 → 최종 후보 {len(최종후보)}개")
print(f"미확보 금액 합계: {최종후보['미확보금액'].sum()/1e8:,.0f}억원 (6개월)")

display(최종후보.assign(실제_억=lambda d: d["amt"] / 1e8,
                     시장_억=lambda d: d["추정시장규모"] / 1e8,
                     미확보_억=lambda d: d["미확보금액"] / 1e8)
        [["행정구역명", "TP_BUZ_NM", "업소수", "실제_억", "시장_억",
          "미확보_억", "침투지수", "파레토", "진입횟수"]])


In [ ]:
# 개별 업종이 아니라 지역 전체가 약한 곳을 따로 찾는다
#   8개 업종 모두 데이터가 있는 지역만 (일부 업종만 있는 곳은 평균이 왜곡된다)
지역종합 = (회귀결과.groupby("행정구역명")
         .agg(업종수=("침투지수", "size"),
              평균침투=("침투지수", "mean"),
              백미만=("침투지수", lambda s: (s < 100).sum()),
              총소비=("amt", "sum")))
지역종합 = 지역종합[지역종합["업종수"] >= 8]

display(지역종합.nsmallest(10, "평균침투")
        .assign(총소비_억=lambda d: d["총소비"] / 1e8)
        [["업종수", "평균침투", "백미만", "총소비_억"]])


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6), facecolor=바탕)
ax.set_facecolor(바탕)

# 배경: 전체 후보풀
ax.scatter(후보["미확보금액"] / 1e8, 후보["침투지수"],
           s=22, color=파랑, alpha=0.22, edgecolor="none", zorder=2)

# 최종 후보 — 업종별로 색을 나눈다
업종색 = {"슈퍼 마켓": 주황, "일반한식": 초록}
for 업종, c in 업종색.items():
    d_ = 최종후보[최종후보["TP_BUZ_NM"] == 업종]
    ax.scatter(d_["미확보금액"] / 1e8, d_["침투지수"], s=110,
               color=c, edgecolor=바탕, lw=2, zorder=5, label=업종)

for _, r in 최종후보.iterrows():
    ax.annotate(r["행정구역명"].split()[-1],
                (r["미확보금액"] / 1e8, r["침투지수"]),
                textcoords="offset points", xytext=(11, -3),
                fontsize=10, color=잉크, fontweight="bold")

ax.axhline(100, color=흐림, ls="--", lw=1)
ax.set_title("최종 후보 8개",
             color=잉크, fontsize=14, fontweight="bold", loc="left", pad=14)
ax.set_xlabel("미확보 금액 (억원) →  놓치고 있는 돈", color=보조, fontsize=10)
ax.set_ylabel("침투지수 ↓  낮을수록 BC가 약함", color=보조, fontsize=10)
ax.tick_params(colors=흐림, labelsize=9.5, length=0)
ax.grid(color=격자, lw=1)
ax.set_axisbelow(True)
for s_ in ["top", "right"]:
    ax.spines[s_].set_visible(False)
for s_ in ["left", "bottom"]:
    ax.spines[s_].set_color(축선)
ax.legend(frameon=False, loc="upper left", fontsize=10, labelcolor=보조)
plt.tight_layout()
plt.show()


검증 1. LOO — 자기 자신을 빼고 예측



In [ ]:
# "그 지역이 모델 학습에 참여했으니 자기 자신을 잘 맞추는 것 아닌가?"라는 의심을 검증한다.
# 227번 재적합할 필요는 없다. OLS에는 PRESS 잔차라는 정확한 지름길이 있어서
# '그 관측치를 빼고 적합했을 때의 예측 오차'를 한 번의 적합으로 얻을 수 있다.
loo목록 = []
for 업종코드, d in 회귀결과.groupby("TP_BUZ_NO"):
    d = d.reset_index(drop=True)
    모델 = sm.OLS(d["log_소비"], sm.add_constant(d[설명변수])).fit()
    영향 = 모델.get_influence()

    d = d.assign(LOO잔차=영향.resid_press,        # = 잔차 / (1 - 레버리지)
                 레버리지=영향.hat_matrix_diag)
    d["LOO지수"] = np.exp(d["LOO잔차"]) * 100
    loo목록.append(d)

회귀결과 = pd.concat(loo목록, ignore_index=True)

print(f"기본 침투지수 vs LOO지수 상관: {회귀결과['침투지수'].corr(회귀결과['LOO지수']):.4f}")
print(f"두 지수 차이 평균: {(회귀결과['침투지수'] - 회귀결과['LOO지수']).abs().mean():.2f}p")
print(f"레버리지 평균 {회귀결과['레버리지'].mean():.4f} / 최대 {회귀결과['레버리지'].max():.3f}")
print("→ 상관이 1에 가까우면 자기 자신이 모델을 끌어당긴 효과가 없다는 뜻")


검증 2. 시간 분할 — 전반기와 후반기가 같은 답을 주는가



In [ ]:
# 1~3월 데이터만으로 침투지수를 만들고, 4~6월로도 따로 만들어 비교한다.
# 두 값이 비슷하면 저침투가 일시적 변동이 아니라 구조적 특성이라는 뜻이다.
성인 = ["2", "3", "4", "5", "6"]
제외업종 = ["8002", "8003", "4004"]

반기 = bc[bc["GENDER_CD"].isin(["1", "2", "3"]) & ~bc["TP_BUZ_NO"].isin(제외업종)].copy()
반기["반기"] = np.where(반기["STRD_YYMM"] <= "202603", "전반", "후반")
반기소비 = 반기.groupby(["행정구역명", "TP_BUZ_NO", "반기"], as_index=False)["amt"].sum()

# 업소수·인구는 6개월간 고정이므로 그대로 쓴다
기준정보 = 회귀결과[["행정구역명", "TP_BUZ_NO", "업소수", "성인인구"]].drop_duplicates()

반기지수 = {}
for 구간, dd in 반기소비.groupby("반기"):
    z = dd.merge(기준정보, on=["행정구역명", "TP_BUZ_NO"])
    z = z[z["amt"] > 0]
    모음 = []
    for 업종코드, d in z.groupby("TP_BUZ_NO"):
        d = d.assign(ls=np.log(d["업소수"]), lp=np.log(d["성인인구"]), la=np.log(d["amt"]))
        m = sm.OLS(d["la"], sm.add_constant(d[["ls", "lp"]])).fit()
        모음.append(d.assign(지수=np.exp(m.resid) * 100)[["행정구역명", "TP_BUZ_NO", "지수"]])
    반기지수[구간] = pd.concat(모음).set_index(["행정구역명", "TP_BUZ_NO"])["지수"]

전, 후 = 반기지수["전반"], 반기지수["후반"]
공통 = 전.index.intersection(후.index)
print(f"전반기 침투지수 vs 후반기 침투지수 상관: {np.corrcoef(전[공통], 후[공통])[0, 1]:.4f}  (n={len(공통)})")
print("→ 1에 가까우면 저침투가 구조적. 일시적 변동이면 상관이 낮게 나온다")


검증 3. Random Forest — 선형 가정을 버려도 같은가



In [ ]:
from sklearn.ensemble import RandomForestRegressor

# 로그선형 관계를 가정하지 않는 모델로도 같은 지역이 이상치로 잡히는지 본다.
# oob_score=True로 '학습에 안 쓴 샘플'에 대한 예측(OOB)을 쓰므로 과적합 걱정이 없다.
rf목록 = []
for 업종코드, d in 회귀결과.groupby("TP_BUZ_NO"):
    d = d.reset_index(drop=True)
    rf = RandomForestRegressor(n_estimators=400, min_samples_leaf=5,
                               random_state=42, oob_score=True)
    rf.fit(d[설명변수], d["log_소비"])
    d["RF지수"] = np.exp(d["log_소비"] - rf.oob_prediction_) * 100
    rf목록.append(d)

회귀결과 = pd.concat(rf목록, ignore_index=True)
print(f"OLS 침투지수 vs RF지수 상관: {회귀결과['침투지수'].corr(회귀결과['RF지수']):.4f}")
print("→ 비선형 모델에서도 같은 지역이 낮게 나오면 선형 가정 탓이 아니라는 뜻")


검증 4. 부트스트랩 — 표본을 바꿔도 후보가 유지되는가



In [ ]:
# 지역을 복원추출로 다시 뽑아 모델을 500번 새로 적합하고,
# 매번 상위 10 후보를 뽑아 각 조합이 몇 번 등장하는지 센다.
#
# ⚠️ 순위를 '침투지수'만으로 매기면 안 된다.
#    우리 후보 선정은 침투지수와 미확보금액 두 축을 함께 보기 때문에,
#    침투지수만으로 줄 세우면 금액 축으로 들어온 일반한식 후보가 전부 0%로 나온다.
rng = np.random.RandomState(42)
반복 = 500
등장 = {}

for b in range(반복):
    예측모음 = []
    for 업종코드, d in 회귀결과.groupby("TP_BUZ_NO"):
        d = d.reset_index(drop=True)
        표본 = d.iloc[rng.randint(0, len(d), len(d))]          # 지역 복원추출
        m = sm.OLS(표본["log_소비"], sm.add_constant(표본[설명변수])).fit()
        예측 = m.predict(sm.add_constant(d[설명변수], has_constant="add"))
        예측모음.append(d.assign(지수=np.exp(d["log_소비"] - 예측) * 100))

    bb = pd.concat(예측모음, ignore_index=True)
    bb = bb[(bb["추정시장규모"] >= bb["시장중앙값"]) & (bb["지수"] < 100)].copy()
    if not len(bb):
        continue
    bb["미확보"] = bb["추정시장규모"] - bb["amt"]

    정규 = lambda x: (x - x.min()) / (x.max() - x.min())
    bb["점수"] = 0.5 * 정규(bb["미확보"]) + 0.5 * 정규(100 - bb["지수"])   # 두 축 균형

    for _, r in bb.nlargest(10, "점수").iterrows():
        키 = (r["행정구역명"], r["TP_BUZ_NM"])
        등장[키] = 등장.get(키, 0) + 1

부트결과 = (pd.DataFrame([{"행정구역명": k[0], "TP_BUZ_NM": k[1], "등장률": v / 반복 * 100}
                      for k, v in 등장.items()])
         .sort_values("등장률", ascending=False))
display(부트결과.head(16))


검증 5. 예측 오차 밴드로 후보 등급 나누기



In [ ]:
from sklearn.model_selection import KFold

# CV 방식으로 업종별 예측 오차를 구한다.
# 이 오차보다 편차가 작으면 "모델이 원래 그 정도는 틀린다"는 범위 안이라
# 저침투라고 단정하기 어렵다.
밴드 = {}
for 업종코드, d in 회귀결과.groupby("TP_BUZ_NO"):
    d = d.reset_index(drop=True)
    오차 = []
    for tr, te in KFold(5, shuffle=True, random_state=42).split(d):
        X_tr = sm.add_constant(d.loc[tr, 설명변수])
        X_te = sm.add_constant(d.loc[te, 설명변수], has_constant="add")
        m = sm.OLS(d.loc[tr, "log_소비"], X_tr).fit()
        오차.append(np.sqrt(np.mean((d.loc[te, "log_소비"] - m.predict(X_te)) ** 2)))
    밴드[업종코드] = np.mean(오차)

밴드표 = pd.DataFrame([{"TP_BUZ_NO": k, "CV_RMSE": v,
                     "정상범위_하한": np.exp(-v) * 100,
                     "정상범위_상한": np.exp(v) * 100} for k, v in 밴드.items()])
밴드표["TP_BUZ_NM"] = 밴드표["TP_BUZ_NO"].map(
    회귀결과.drop_duplicates("TP_BUZ_NO").set_index("TP_BUZ_NO")["TP_BUZ_NM"])

with pd.option_context("display.float_format", "{:.1f}".format):
    display(밴드표[["TP_BUZ_NM", "CV_RMSE", "정상범위_하한", "정상범위_상한"]]
            .sort_values("CV_RMSE"))


In [ ]:
# 최종 후보에 검증 결과를 모두 붙이고 등급을 매긴다
검증표 = 최종후보[["행정구역명", "TP_BUZ_NO", "TP_BUZ_NM", "침투지수", "미확보금액"]].copy()

붙이기 = 회귀결과.set_index(["행정구역명", "TP_BUZ_NO"])
검증표["LOO"] = [붙이기.loc[(g, c), "LOO지수"] for g, c in zip(검증표["행정구역명"], 검증표["TP_BUZ_NO"])]
검증표["RF"]  = [붙이기.loc[(g, c), "RF지수"]  for g, c in zip(검증표["행정구역명"], 검증표["TP_BUZ_NO"])]
검증표["전반"] = [전.get((g, c), np.nan) for g, c in zip(검증표["행정구역명"], 검증표["TP_BUZ_NO"])]
검증표["후반"] = [후.get((g, c), np.nan) for g, c in zip(검증표["행정구역명"], 검증표["TP_BUZ_NO"])]
검증표 = 검증표.merge(부트결과, on=["행정구역명", "TP_BUZ_NM"], how="left")

# 침투 부족이 모델 오차의 몇 배인가 (음수일수록 강함)
검증표["시그마"] = [np.log(i / 100) / 밴드[c]
                for i, c in zip(검증표["침투지수"], 검증표["TP_BUZ_NO"])]

def 등급(r):
    if r["시그마"] < -2 and r["등장률"] >= 95:  return "A  확실"
    if r["시그마"] < -1.5:                      return "B  양호"
    return "C  규모로 선정 — 편차는 오차 범위"

검증표["등급"] = 검증표.apply(등급, axis=1)

with pd.option_context("display.float_format", "{:.1f}".format):
    display(검증표.sort_values(["등급", "시그마"])
            [["등급", "행정구역명", "TP_BUZ_NM", "침투지수", "LOO", "RF",
              "전반", "후반", "등장률", "시그마"]])


In [ ]:
# ── 시장규모와 격차를 하나의 모델에서 뽑는다 ──
# 침투지수가 '실제 ÷ 모델예측'이므로, 시장규모와 격차도 같은 모델예측을 기준으로 삼아야
# 기준이 갈리지 않는다. 기존에 쓰던 '업소수 × 전국평균'은 점포 수만 보고 인구를 무시했다.
#
#   모델예측  = 이 지역·업종이라면 나왔어야 할 BC 소비 (점포 수 + 인구 반영)
#   격차금액  = 모델예측 − 실제 = 놓치고 있는 돈
#   침투지수  = 실제 ÷ 모델예측 × 100
회귀결과["모델예측"] = 회귀결과["amt"] / 회귀결과["침투지수"] * 100
회귀결과["격차금액"] = 회귀결과["모델예측"] - 회귀결과["amt"]

# 시장규모 필터도 모델예측 기준으로
#   모델예측은 그 지역 자신의 소비에 거의 영향받지 않는다(LOO 상관 0.9988)
#   따라서 종속변수로 필터링하는 문제가 생기지 않는다
회귀결과["시장중앙값"] = 회귀결과.groupby("TP_BUZ_NO")["모델예측"].transform("median")

후보 = 회귀결과[(회귀결과["모델예측"] >= 회귀결과["시장중앙값"]) &
             (회귀결과["침투지수"] < 100)].copy()

print(f"전체 {len(회귀결과)}개 → 시장규모 상위 50% & 침투지수 100 미만: {len(후보)}개")
print(f"격차금액 합계: {후보['격차금액'].sum()/1e8:,.0f}억 (6개월)")


In [ ]:
# 파레토 최적 재계산
s = 후보.sort_values("격차금액", ascending=False).reset_index(drop=True)
최저 = np.inf
프론티어 = []
for i, r in s.iterrows():
    if r["침투지수"] < 최저:
        프론티어.append(i)
        최저 = r["침투지수"]
파레토 = s.loc[프론티어].sort_values("침투지수")

display(파레토.assign(실제_억=lambda d: d["amt"] / 1e8,
                    시장_억=lambda d: d["모델예측"] / 1e8,
                    격차_억=lambda d: d["격차금액"] / 1e8)
        [["행정구역명", "TP_BUZ_NM", "업소수", "실제_억", "시장_억", "격차_억", "침투지수"]])


In [ ]:
# ── 사분면 ──
# 시장규모: 모델예측의 업종 내 백분위 — 업종마다 규모가 10배 이상 달라 업종 안에서 비교한다
# 격차   : 침투지수 100 — "모델 예측대로 나왔는가"라는 자연스러운 분기점
회귀결과["시장백분위"] = 회귀결과.groupby("TP_BUZ_NO")["모델예측"].rank(pct=True) * 100
회귀결과["시장큼"] = 회귀결과["시장백분위"] > 50
회귀결과["격차큼"] = 회귀결과["침투지수"] < 100

사분면이름 = {(True, True): "최우선 공략", (True, False): "유지관리",
          (False, True): "잠재력 관찰", (False, False): "저우선순위"}
회귀결과["사분면"] = [사분면이름[(a, b)] for a, b in zip(회귀결과["시장큼"], 회귀결과["격차큼"])]

# ── 신뢰 등급 ──
# 업종별 CV 오차(밴드)보다 편차가 얼마나 큰가. 앞 검증 셀을 안 돌렸으면 여기서 계산한다
if "밴드" not in globals():
    from sklearn.model_selection import KFold
    밴드 = {}
    for 업종코드, d in 회귀결과.groupby("TP_BUZ_NO"):
        d = d.reset_index(drop=True)
        오차 = []
        for tr, te in KFold(5, shuffle=True, random_state=42).split(d):
            m = sm.OLS(d.loc[tr, "log_소비"], sm.add_constant(d.loc[tr, 설명변수])).fit()
            p = m.predict(sm.add_constant(d.loc[te, 설명변수], has_constant="add"))
            오차.append(np.sqrt(np.mean((d.loc[te, "log_소비"] - p) ** 2)))
        밴드[업종코드] = np.mean(오차)

회귀결과["시그마"] = np.log(회귀결과["침투지수"] / 100) / 회귀결과["TP_BUZ_NO"].map(밴드)
# 격차 작음 쪽에는 신뢰 등급이 의미가 없으므로 "-"
회귀결과["신뢰"] = np.select(
    [~회귀결과["격차큼"], 회귀결과["시그마"] <= -2, 회귀결과["시그마"] <= -1],
    ["-", "강", "중"], default="약")

# ── 거래월수 — 6개월 중 거래가 있었던 달 수 (데이터 부족 표시용) ──
제외3 = ["8002", "8003", "4004"]
b = bc[bc["GENDER_CD"].isin(["1", "2", "3"]) & ~bc["TP_BUZ_NO"].isin(제외3)]
거래월 = (b[b["amt"] > 0].groupby(["행정구역명", "TP_BUZ_NO"])["STRD_YYMM"]
        .nunique().rename("거래월수").reset_index())

# 셀을 다시 돌려도 컬럼이 중복되지 않게 먼저 지운다
회귀결과 = 회귀결과.drop(columns=[c for c in ["거래월수"] if c in 회귀결과.columns])
회귀결과 = 회귀결과.merge(거래월, on=["행정구역명", "TP_BUZ_NO"], how="left")

print(회귀결과["사분면"].value_counts().to_string())
print()
print("6개월 미만 거래 조합:", (회귀결과["거래월수"] < 6).sum(), "개")
